# 5. cVAE + Adversarial Batch Correction (no KL)

Batch correction COMPASS PT embeddings via **cVAE + adversarial discriminator**.

**Architecture:**
```
Encoder (batch-FREE):  4224 -> 512 -> 256 -> (mu, logvar) -> z
Decoder (batch-COND):  (z + batch_onehot) -> 256 -> 512 -> 4224
Discriminator (GRL):   GRL(z) -> 128 -> n_batches
Loss:  MSE_recon + CE(batch_pred, batch_true)  [GRL handles lambda]
```

No KL divergence (it destroyed biology). GRL forces encoder to hide batch info.

**Tuned defaults:** lr=5e-4, lambda_adv=0.1, warmup=40%, grad_clip=0.5, label_smoothing=0.1

## 0. Environment Setup

In [ ]:
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger
from matplotlib.lines import Line2D
from sklearn.metrics import silhouette_score
from umap import UMAP

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import CVAEAdvCorrector, CVAEAdvConfig

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}, PyTorch: {}", DEVICE, torch.__version__)

## 1. Mount Drive & Define Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_adv')
DATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')

SPLIT_COL = "Split_dummy"
ADV_KEY = SCVI_CORRECTED_KEY + "_adv"

logger.info("Source: {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])

## 2. UMAP Helper

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.0)

def plot_umap_panel(coords, labels, title, ax, palette=None):
    """Scatter UMAP colored by labels."""
    unique = sorted(labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique))
    cmap = dict(zip(unique, palette))
    colors = [cmap[l] for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6,
               edgecolors="none", rasterized=True)
    ax.set_title(title, fontsize=10, fontweight="bold")
    if len(unique) <= 12:
        handles = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cmap[l], markersize=7, label=str(l))
                   for l in unique]
        ax.legend(handles=handles, fontsize=6, loc="best",
                  framealpha=0.8, edgecolor="#ccc")

## 3. Process All Cohorts

For each cohort:
1. Load from `interim_PT`
2. Add `Split_dummy` (60/20/20)
3. Fit Adv-cVAE on train -> transform all
4. Single-batch cohorts: copy raw embeddings as-is
5. Save to `processed_adv/`

In [ ]:
# Uses tuned defaults from CVAEAdvConfig:
# lr=5e-4, lambda_adv=0.1, n_epochs=100, warmup=0.4,
# grad_clip=0.5, label_smoothing=0.1
adv_config = CVAEAdvConfig(latent_dim=64, seed=SEED)

all_histories = {}

for cohort_name in COHORT_CANCER_CODE:
    src_path  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    dest_path = DATA_PROCESSED_ADV  / f"{cohort_name}.adata.zarr"

    if not src_path.exists():
        logger.warning("Missing: {}", cohort_name)
        continue
    if dest_path.exists():
        logger.info("Already done, skipping: {}", cohort_name)
        continue

    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    adata = load_cohort(src_path)
    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    if COMPASS_PT_EMBEDDING_KEY not in adata.obsm:
        logger.error("Missing embedding, skipping")
        del adata; continue

    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)

    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Split: train={}, test={}, NaN={}",
        train_mask.sum(), test_mask.sum(),
        adata.n_obs - train_mask.sum() - test_mask.sum(),
    )

    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch_labels_all = adata.obs[BATCH_COL].astype(str).values
    n_unique_batches = len(set(batch_labels_all))

    if n_unique_batches < 2:
        logger.info("Single batch -> copying raw embeddings as-is")
        adata.obsm[ADV_KEY] = emb_all
    else:
        corrector = CVAEAdvCorrector(config=adv_config)
        corrector.fit(
            X=emb_all[train_mask],
            batch_labels=batch_labels_all[train_mask],
        )
        all_histories[cohort_name] = corrector.history_
        corrected = corrector.transform(emb_all)
        adata.obsm[ADV_KEY] = corrected

        logger.info("Corrected shape: {}, dtype: {}", corrected.shape, corrected.dtype)
        assert corrected.shape[1] == adv_config.latent_dim
        assert np.isfinite(corrected).all(), "NaN/Inf detected!"

    save_adata_zarr(adata, dest_path)
    logger.info("Saved: {}", dest_path)
    del adata
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed.")

## 4. Training Loss Curves

In [ ]:
if not all_histories:
    logger.warning("No histories to plot.")
else:
    for cname, hist in all_histories.items():
        fig, axes = plt.subplots(1, 4, figsize=(22, 4))
        fig.suptitle(f"{cname} -- Adv-cVAE Training", fontsize=13, fontweight="bold", y=1.03)
        epochs = range(1, len(hist.total) + 1)

        axes[0].plot(epochs, hist.recon, color='#2196F3', lw=1.5)
        axes[0].set_title("Reconstruction (MSE)")
        axes[0].set_xlabel("Epoch")

        axes[1].plot(epochs, hist.adv, color='#FF5722', lw=1.5)
        axes[1].set_title("Adversarial Loss (CE)")
        axes[1].set_xlabel("Epoch")

        axes[2].plot(epochs, hist.disc_accuracy, color='#9C27B0', lw=1.5)
        axes[2].set_title("Disc Accuracy")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylim(0, 1)

        axes[3].plot(epochs, hist.lambda_schedule, color='#4CAF50', lw=1.5)
        axes[3].set_title("Lambda Schedule")
        axes[3].set_xlabel("Epoch")

        plt.tight_layout()
        plt.show()

## 5. Before / After UMAP

UMAP for the first available multi-batch cohort.

In [ ]:
test_cohort = None
for name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    if p.exists():
        ad_tmp = load_cohort(p)
        if ad_tmp.obs[BATCH_COL].nunique() > 1:
            test_cohort = name
        del ad_tmp
        if test_cohort: break

if test_cohort:
    adata = load_cohort(DATA_PROCESSED_ADV / f"{test_cohort}.adata.zarr")
    emb_raw = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    umap_raw = UMAP(n_components=2, random_state=SEED).fit_transform(emb_raw)
    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)
    umap_corr = UMAP(n_components=2, random_state=SEED).fit_transform(emb_corr)

    batch = adata.obs[BATCH_COL].astype(str)
    diag  = adata.obs[DIAGNOSIS_COL].astype(str)
    batch_pal = sns.color_palette("husl", n_colors=batch.nunique())
    diag_pal  = sns.color_palette("Set2", n_colors=diag.nunique())

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{test_cohort} -- Before vs After Adv-cVAE ({adata.n_obs} samples)",
                 fontsize=14, fontweight="bold", y=1.02)
    plot_umap_panel(umap_raw,  batch, "Before -- Batch",     axes[0, 0], batch_pal)
    plot_umap_panel(umap_corr, batch, "After -- Batch",      axes[0, 1], batch_pal)
    plot_umap_panel(umap_raw,  diag,  "Before -- Diagnosis", axes[1, 0], diag_pal)
    plot_umap_panel(umap_corr, diag,  "After -- Diagnosis",  axes[1, 1], diag_pal)
    for ax in axes.flat:
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    plt.tight_layout(); plt.show()
    del adata
else:
    logger.warning("No multi-batch cohort found.")

## 6. Silhouette Scores

In [ ]:
rows = []
for name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    if not p.exists(): continue
    adata = load_cohort(p)
    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)
    batch = adata.obs[BATCH_COL].astype(str).values
    diag  = adata.obs[DIAGNOSIS_COL].astype(str).values
    mb = len(set(batch)) > 1; md_ = len(set(diag)) > 1
    rows.append({
        "Cohort": name, "Samples": adata.n_obs,
        "Sil_Batch_Before": round(silhouette_score(emb_raw, batch), 4) if mb else np.nan,
        "Sil_Batch_After":  round(silhouette_score(emb_corr, batch), 4) if mb else np.nan,
        "Sil_Diag_Before":  round(silhouette_score(emb_raw, diag), 4) if md_ else np.nan,
        "Sil_Diag_After":   round(silhouette_score(emb_corr, diag), 4) if md_ else np.nan,
    })
    del adata
if rows:
    df_sil = pd.DataFrame(rows)
    df_sil["Sil_Batch_D"] = df_sil["Sil_Batch_After"] - df_sil["Sil_Batch_Before"]
    df_sil["Sil_Diag_D"]  = df_sil["Sil_Diag_After"]  - df_sil["Sil_Diag_Before"]
    display(df_sil)
    print("Sil_Batch_D < 0 = better mixing (good)")
    print("Sil_Diag_D >= 0 = biology preserved (good)")
else:
    logger.warning("No cohorts found.")

## 7. Immotion Control -- Over-correction Check

ccRCC ICI and TKI from the same lab. Silhouette by `Cohort` should NOT change.

In [ ]:
immotion_cohorts = [
    "PUB_ccRCC_Immotion150_and_151_ICI",
    "PUB_ccRCC_Immotion150_and_151_TKI",
]
for cname in immotion_cohorts:
    p = DATA_PROCESSED_ADV / f"{cname}.adata.zarr"
    if not p.exists(): continue
    adata = load_cohort(p)
    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)
    if "Cohort" in adata.obs.columns:
        cohort_labels = adata.obs["Cohort"].astype(str).values
        if len(set(cohort_labels)) > 1:
            sil_raw  = silhouette_score(emb_raw, cohort_labels)
            sil_corr = silhouette_score(emb_corr, cohort_labels)
            print(f"{cname}:")
            print(f"  Sil_Cohort Before: {sil_raw:.4f}")
            print(f"  Sil_Cohort After:  {sil_corr:.4f}")
            print(f"  Delta: {sil_corr - sil_raw:.4f}  (should be ~0)")
        else:
            print(f"{cname}: single Cohort value")
    else:
        print(f"{cname}: no 'Cohort' column")
    del adata

## 8. Mariathasan Permutation Test

BLCA has 1 batch. Random batch labels -> train -> silhouette delta should be ~0.

In [ ]:
blca_name = "PUB_BLCA_Mariathasan_EGAS00001002556"
blca_path = DATA_INTERIM_PT_DIR / f"{blca_name}.adata.zarr"

if blca_path.exists():
    adata = load_cohort(blca_path)
    emb = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    n = adata.n_obs

    batch_pool = []
    for cname in COHORT_CANCER_CODE:
        if cname == blca_name: continue
        p = DATA_INTERIM_PT_DIR / f"{cname}.adata.zarr"
        if not p.exists(): continue
        ad_tmp = load_cohort(p)
        batch_pool.extend(ad_tmp.obs[BATCH_COL].astype(str).values.tolist())
        del ad_tmp
    batch_pool = np.array(batch_pool)
    logger.info("Pool: {} samples, {} unique", len(batch_pool), len(set(batch_pool)))

    N_PERMS = 5
    perm_results = []
    for i in range(N_PERMS):
        rng = np.random.RandomState(SEED + i)
        fake_labels = rng.choice(batch_pool, size=n, replace=True)
        n_fake = len(set(fake_labels))
        if n_fake < 2: continue

        cfg_perm = CVAEAdvConfig(latent_dim=64, seed=SEED + i)
        corr = CVAEAdvCorrector(config=cfg_perm)
        corr.fit(X=emb, batch_labels=fake_labels)
        emb_perm = corr.transform(emb)

        sil_raw  = silhouette_score(emb, fake_labels)
        sil_perm = silhouette_score(emb_perm, fake_labels)
        perm_results.append({"perm": i, "n_fake": n_fake,
            "sil_before": round(sil_raw, 4), "sil_after": round(sil_perm, 4),
            "delta": round(sil_perm - sil_raw, 4)})

    if perm_results:
        df_perm = pd.DataFrame(perm_results)
        display(df_perm)
        print(f"Mean delta: {df_perm['delta'].mean():.4f} (should be ~0)")
    del adata
else:
    logger.warning("BLCA not found")

## 9. Comparison: old cVAE (KL) vs Adversarial cVAE

Head-to-head if `data/processed/` (notebook 4) results are available.

In [ ]:
rows_cmp = []
for cohort_name in COHORT_CANCER_CODE:
    old_p = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    new_p = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    if not old_p.exists() or not new_p.exists(): continue

    ad_old = load_cohort(old_p)
    ad_new = load_cohort(new_p)
    batch = ad_old.obs[BATCH_COL].astype(str).values
    diag  = ad_old.obs[DIAGNOSIS_COL].astype(str).values
    mb = len(set(batch)) > 1; md_ = len(set(diag)) > 1

    emb_raw = np.asarray(ad_old.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    sil_b_raw = silhouette_score(emb_raw, batch) if mb else np.nan
    sil_d_raw = silhouette_score(emb_raw, diag) if md_ else np.nan

    if SCVI_CORRECTED_KEY in ad_old.obsm:
        e_old = np.asarray(ad_old.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
        sil_b_old = silhouette_score(e_old, batch) if mb else np.nan
        sil_d_old = silhouette_score(e_old, diag) if md_ else np.nan
    else:
        sil_b_old = sil_d_old = np.nan

    if ADV_KEY in ad_new.obsm:
        e_adv = np.asarray(ad_new.obsm[ADV_KEY], dtype=np.float32)
        sil_b_adv = silhouette_score(e_adv, batch) if mb else np.nan
        sil_d_adv = silhouette_score(e_adv, diag) if md_ else np.nan
    else:
        sil_b_adv = sil_d_adv = np.nan

    rows_cmp.append({
        "Cohort": cohort_name,
        "Sil_Batch_Raw": round(sil_b_raw, 4) if not np.isnan(sil_b_raw) else np.nan,
        "Sil_Batch_KL": round(sil_b_old, 4) if not np.isnan(sil_b_old) else np.nan,
        "Sil_Batch_Adv": round(sil_b_adv, 4) if not np.isnan(sil_b_adv) else np.nan,
        "Sil_Diag_Raw": round(sil_d_raw, 4) if not np.isnan(sil_d_raw) else np.nan,
        "Sil_Diag_KL": round(sil_d_old, 4) if not np.isnan(sil_d_old) else np.nan,
        "Sil_Diag_Adv": round(sil_d_adv, 4) if not np.isnan(sil_d_adv) else np.nan,
    })
    del ad_old, ad_new

if rows_cmp:
    df_cmp = pd.DataFrame(rows_cmp)
    display(df_cmp)
    print("Sil_Batch: lower = better (mixed)")
    print("Sil_Diag:  higher = better (biology)")
    print("Ideal: Adv has Sil_Batch < KL AND Sil_Diag > KL")
else:
    logger.warning("No cohorts in both processed/ and processed_adv/")